# Load Data

In [2]:
%run "..\scripts\load_daily_variables.py"

Loading Variables: 100%|██████████| 27/27 [07:25<00:00, 16.49s/it]  


In [5]:
sector_etfs = """SPY
QQQ
XLF
XLK
XLU
XLB
XLP
XLV
XLY
XLI
IWM
MDY
XLC
IBB
XBI
XLE
MJ
VNQ
TAN
DRIV
MCHI
BIB
PBW
FDN
IGV
ITB
XME
XRT
IYT
PEJ
IYZ
ITA
XOP
SMH
FXI""".split('\n')

major_indices = ['SPY', 'QQQ']

market_cap_etfs = ['SPY', 'IWM', 'MDY']

# Plot Relative Rotation Graph

In [6]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px

# RRG controls. Dates are inclusive; use None for the first/latest available date.
BENCHMARK = "SPY"
START_DATE = "2026-02-27"
END_DATE = None
OBSERVATION_FREQUENCY = None # W-FRI  # Use None for daily observations.
RS_LOOKBACK = 50
MOMENTUM_LOOKBACK = 10
TAIL_LENGTH = 14 # 20  # Use None to show every observation in the selected date range.
symbols_list = sector_etfs
RRG_SYMBOLS = [symbol for symbol in symbols_list if symbol != BENCHMARK]


def close_series(symbol):
    """Return a clean Close series from the ETF SymbolData objects loaded above."""
    frame = etfs[symbol].df
    if "Close" not in frame.columns:
        raise KeyError(f"{symbol} has no Close column")

    close = pd.to_numeric(frame["Close"], errors="coerce").copy()
    close.index = pd.to_datetime(close.index)
    return close.groupby(level=0).last().sort_index().rename(symbol)


available = [symbol for symbol in [BENCHMARK, *RRG_SYMBOLS] if symbol in etfs]
missing = sorted(set([BENCHMARK, *RRG_SYMBOLS]) - set(available))
if BENCHMARK not in available:
    raise KeyError(f"Benchmark {BENCHMARK!r} was not found in the loaded ETFs")
if missing:
    print("Skipped symbols not present in the loaded ETF data:", ", ".join(missing))

prices = pd.concat([close_series(symbol) for symbol in available], axis=1).sort_index()
if OBSERVATION_FREQUENCY:
    prices = prices.resample(OBSERVATION_FREQUENCY).last()

# JdK's exact indicators are proprietary. This common RRG approximation normalizes
# price-relative trend and its momentum around the neutral value of 100.
relative_strength = prices.div(prices[BENCHMARK], axis=0)
rs_ratio = 100 * relative_strength.div(
    relative_strength.rolling(RS_LOOKBACK, min_periods=RS_LOOKBACK).mean()
)
rs_momentum = 100 * rs_ratio.div(rs_ratio.shift(MOMENTUM_LOOKBACK))

start = pd.Timestamp(START_DATE) if START_DATE is not None else rs_ratio.index.min()
end = pd.Timestamp(END_DATE) if END_DATE is not None else rs_ratio.index.max()
if start > end:
    raise ValueError("START_DATE must be on or before END_DATE")

plot_symbols = [symbol for symbol in RRG_SYMBOLS if symbol in available]
rrg = pd.concat(
    {symbol: pd.DataFrame({"RS-Ratio": rs_ratio[symbol], "RS-Momentum": rs_momentum[symbol]})
     for symbol in plot_symbols},
    names=["Symbol", "Date"],
).dropna().reset_index()
rrg = rrg.loc[rrg["Date"].between(start, end)].copy()
if rrg.empty:
    raise ValueError(
        "No RRG observations fall in the selected range. Choose later dates or "
        "reduce RS_LOOKBACK/MOMENTUM_LOOKBACK."
    )
if TAIL_LENGTH is not None:
    rrg = rrg.groupby("Symbol", group_keys=False).tail(TAIL_LENGTH)

padding = max(
    1.0,
    0.08 * max(rrg["RS-Ratio"].max() - rrg["RS-Ratio"].min(),
               rrg["RS-Momentum"].max() - rrg["RS-Momentum"].min()),
)
x_min, x_max = min(rrg["RS-Ratio"].min() - padding, 99), max(rrg["RS-Ratio"].max() + padding, 101)
y_min, y_max = min(rrg["RS-Momentum"].min() - padding, 99), max(rrg["RS-Momentum"].max() + padding, 101)

fig = go.Figure()
quadrants = [
    (100, x_max, 100, y_max, "rgba(76,175,80,0.10)"),    # Leading
    (100, x_max, y_min, 100, "rgba(255,193,7,0.10)"),    # Weakening
    (x_min, 100, y_min, 100, "rgba(244,67,54,0.10)"),    # Lagging
    (x_min, 100, 100, y_max, "rgba(33,150,243,0.10)"),   # Improving
]
for x0, x1, y0, y1, color in quadrants:
    fig.add_shape(type="rect", x0=x0, x1=x1, y0=y0, y1=y1,
                  fillcolor=color, line_width=0, layer="below")

colors = px.colors.qualitative.Alphabet
for index, (symbol, trail) in enumerate(rrg.groupby("Symbol", sort=False)):
    trail = trail.sort_values("Date")
    marker_sizes = [6] * max(len(trail) - 1, 0) + [12]
    labels = [""] * max(len(trail) - 1, 0) + [symbol]
    fig.add_trace(go.Scatter(
        x=trail["RS-Ratio"],
        y=trail["RS-Momentum"],
        mode="lines+markers+text",
        name=symbol,
        text=labels,
        textposition="top center",
        line=dict(color=colors[index % len(colors)], width=2),
        marker=dict(color=colors[index % len(colors)], size=marker_sizes),
        customdata=np.column_stack([
            trail["Symbol"],
            trail["Date"].dt.strftime("%Y-%m-%d"),
        ]),
        hovertemplate=(
            "Symbol: %{customdata[0]}<br>"
            "Date: %{customdata[1]}<br>"
            "RS-Ratio: %{x:.2f}<br>"
            "RS-Momentum: %{y:.2f}<extra></extra>"
        ),
    ))

trace_count = len(fig.data)
fig.update_layout(
    updatemenus=[dict(
        type="buttons",
        direction="right",
        x=1,
        xanchor="right",
        y=1.12,
        buttons=[
            dict(
                label="Show all",
                method="restyle",
                args=[{"visible": [True] * trace_count}],
            ),
            dict(
                label="Hide all",
                method="restyle",
                args=[{"visible": ["legendonly"] * trace_count}],
            ),
        ],
    )]
)

fig.add_vline(x=100, line_color="gray", line_width=1)
fig.add_hline(y=100, line_color="gray", line_width=1)
for text, x, y in [
    ("IMPROVING", 0.02, 0.98), ("LEADING", 0.98, 0.98),
    ("LAGGING", 0.02, 0.02), ("WEAKENING", 0.98, 0.02),
]:
    fig.add_annotation(text=text, x=x, y=y, xref="paper", yref="paper",
                       showarrow=False, xanchor="left" if x < 0.5 else "right",
                       yanchor="top" if y > 0.5 else "bottom",
                       font=dict(size=12, color="gray"))

fig.update_layout(
    title=f"Relative Rotation Graph vs {BENCHMARK} | {start:%Y-%m-%d} to {end:%Y-%m-%d}",
    xaxis_title="RS-Ratio (relative trend)",
    yaxis_title="RS-Momentum (trend momentum)",
    xaxis=dict(range=[x_min, x_max], zeroline=False),
    yaxis=dict(range=[y_min, y_max], zeroline=False),
    template="plotly_white",
    hovermode="closest",
    legend_title_text="Symbol",
    height=800,
)
fig.show()

# ETF Membership Review

## Symbol Import

In [4]:
import requests
from api_keys import finviz_api_key
from io import StringIO

ETF = 'IYZ'

def etf_membership(etf):
    url = f"https://elite.finviz.com/export/screener?v=111&f=etf_heldby_{etf}&ft=5&auth={finviz_api_key}"
    response = requests.get(url)
    result = pd.read_csv(StringIO(response.text))
    return result

etf_membership(ETF)

,No.,Ticker,Company,Sector,Industry,Country,Market Cap,P/E,Price,Change,Volume
0,1,AAOI,Applied Optoelectronics Inc,Technology,Communication Equipment,USA,7568.50,NaN,94.32,4.67%,9739870
1,2,ANET,Arista Networks Inc,Technology,Computer Hardware,USA,227097.20,61.76,180.35,5.46%,8472358
2,3,ASTS,AST SpaceMobile Inc,Technology,Communication Equipment,USA,22891.59,NaN,58.98,0.92%,9758435
3,4,CHTR,Charter Communications Inc,Communication Services,Telecom Services,USA,19395.60,3.77,144.98,2.10%,2827223
4,5,CIEN,CIENA Corp,Technology,Communication Equipment,USA,53372.53,125.50,377.05,1.34%,1948519
5,6,CMCSA,Comcast Corp,Communication Services,Telecom Services,USA,85025.33,7.76,23.96,1.23%,28202430
6,7,CSCO,Cisco Systems Inc,Technology,Communication Equipment,USA,457167.02,37.81,115.99,2.14%,18812041
7,8,ECHO,EchoStar Corp,Communication Services,Telecom Services,USA,24370.41,NaN,84.09,-2.37%,4127410
8,9,GSAT,Globalstar Inc,Communication Services,Telecom Services,USA,10738.42,NaN,83.38,4.11%,2283437
9,10,IRDM,Iridium Communications Inc,Communication Services,Telecom Services,USA,5016.16,53.94,47.34,-0.63%,1307032


## Quant Ratings

In [ ]:
col = daily_quant_rating_df.columns[-2]
etf_members = etf_membership(ETF)

#* Rating > 4
s = daily_quant_rating_df.loc[(daily_quant_rating_df.index.isin(etf_members['Ticker']))
                          & (daily_quant_rating_df[col] >= 4)][col].sort_values(ascending=False)
#* All ratingss
# s = daily_quant_rating_df.loc[(daily_quant_rating_df.index.isin(result['Ticker']))][col].sort_values(ascending=False)

def length_check(s):
    if len(s) > 60:
        return s.head(60)
    else:
        return s
length_check(s)

Symbol
VIAV    4.933243
CIEN    4.927793
ANET    4.551771
TMUS    4.478261
TIGO    4.027174
Name: 2026-07-17, dtype: float64

# Symbol Selection

In [ ]:
import time

SELECTED_ETFS = """IBB
BIB
SMH
XLK
IGV
FDN
MCHI
FXI
ITA
IWM
MDY
XLV
IYZ""".split('\n')

def top_stocks_from_selected_etfs():
    top_stocks = []
    for etf in SELECTED_ETFS:
        etf_members = etf_membership(etf)

        col = daily_quant_rating_df.columns[-2]

        #* Rating > 4
        s = daily_quant_rating_df.loc[(daily_quant_rating_df.index.isin(etf_members['Ticker']))
                                & (daily_quant_rating_df[col] >= 4)][col]
        top_stocks.extend(s.index)
        time.sleep(5)
    return top_stocks

top_stocks = top_stocks_from_selected_etfs()


In [10]:
earnings_estimate_changes = []

for symbol in pd.unique(top_stocks):
    try:
        revenue_actual = float(
            sa.earnings_dict[symbol]["revenue_actual"]["0"][0]["dataitemvalue"]
        )
        revenue_estimate = float(
            sa.earnings_dict[symbol]["revenue_consensus_mean"]["1"][0]["dataitemvalue"]
        )
        percent_change = (
            (revenue_estimate - revenue_actual) / revenue_actual * 100
            if revenue_actual != 0
            else pd.NA
        )
    except (KeyError, IndexError, TypeError, ValueError):
        percent_change = pd.NA

    earnings_estimate_changes.append(
        {
            "Symbol": symbol,
            "Upcoming Earnings Estimate Change (%)": percent_change,
        }
    )

earnings_estimate_change_df = pd.DataFrame(earnings_estimate_changes).set_index('Symbol')
earnings_estimate_change_df

C:\Users\jdejo\AppData\Local\Temp\ipykernel_36040\714930956.py:3: FutureWarning:

unique with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.



,Upcoming Earnings Estimate Change (%)
Symbol,
QURE,50.299755
ROIV,-5.861111
NBIX,8.578144
KRYS,5.757135
ABCL,-6.396392
...,...
CNC,-4.997565
DXCM,8.266557
DVA,2.283832


In [14]:
(
    earnings_estimate_change_df.loc[
        earnings_estimate_change_df['Upcoming Earnings Estimate Change (%)'] > 20
    ].sort_values(by='Upcoming Earnings Estimate Change (%)', ascending=False)
)

,Upcoming Earnings Estimate Change (%)
Symbol,
PRSU,136.380272
PRM,73.415986
LGIH,54.690119
QURE,50.299755
ASC,48.897385
AMC,43.307658
DHT,42.721773
FTDR,42.373541
NXST,41.86187


# ETF Symbol Comparison by Relative Performance

In [26]:
igv = [symbol for symbol in etf_membership('IGV').Ticker.to_list() if symbol in symbols]

In [27]:
from support_functions import relative

df = pd.DataFrame()

for sym in igv:
    s = relative(
        df=symbols[sym].df,
        bm=etfs['IGV'].df['Close'],
        start='2026-03-27',
    ).Relative_Close
    s.name = sym
    df = pd.concat([df, s], axis=1)






In [41]:
df.loc[:, (df.iloc[-1] < df.iloc[-2]) & (df.iloc[-1] > 100)].iloc[-1].sort_values(ascending=False)

BB      218.677
DDOG    190.295
HUT     184.578
TENB    160.430
FIVN    154.987
VRNS    153.125
U       132.540
CIFR    132.014
CLSK    129.173
RBRK    128.638
RIOT    128.105
NTNX    127.554
CVLT    125.877
MANH    122.110
GEN     121.678
AGYS    121.488
ZETA    120.687
RAMP    119.700
PRGS    119.256
FRSH    117.296
ACIW    116.868
MARA    114.747
LIF     113.836
CORZ    111.776
VYX     108.614
ATEN    107.607
NCNO    107.118
QBTS    105.744
ALRM    104.753
TTWO    104.109
DT      100.336
Name: 2026-07-31 00:00:00, dtype: float64